# Redação: ajuste fino completo do BERTimbau (teste com GPU)

Pergunta que este caderno responde: **treinar o BERTimbau inteiro, com todos os dados, chega perto dos 0,60 a 0,73 de QWK que a literatura reporta?** Na rodada curta em CPU (400 redações) deu 0,30.

**Como usar**
- **Kaggle:** Settings, Accelerator, GPU (T4 ou P100) e Internet **ligada**. Depois "Run All".
- **Colab:** Ambiente de execução, Alterar tipo, GPU. Depois "Executar tudo".
- Leva de **30 a 90 minutos**, conforme a GPU (estimativa, não medida). Deixe a aba aberta.
- No fim ele imprime uma tabela e grava `resultado.json`. Me mande a tabela (ou o arquivo).

**O que ele usa:** só o corpus público Essay-BR (revisão fixada, checksum conferido) e o modelo aberto `neuralmind/bert-base-portuguese-cased`. Nenhum dado de aluno do Olena entra.

**Limites a lembrar ao ler o resultado:** são redações de simulado, não de prova; os textos têm direitos de terceiros que não foram verificados para uso em produto; uma rodada, uma semente. Isto é pesquisa, não libera nota no app.

In [ ]:
# Configuração. Para testar o caderno rapidamente (CPU, poucos minutos): SMOKE = True.
SMOKE = False
MODEL = "neuralmind/bert-base-portuguese-cased"
MAXLEN = 512          # a literatura usa até 512 tokens
EPOCHS = 4            # escolhe a melhor época pela validação
LR = 3e-5
BATCH = 8
SEED = 1

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "scikit-learn"], check=True)

In [ ]:
import ast, csv, hashlib, json, random, re, time, unicodedata, urllib.request
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import cohen_kappa_score
from transformers import AutoModel, AutoTokenizer

REVISION = "31e502ff3075e182cdda2560a767be65db677fa5"
EXPECTED_SHA256 = "f160cd1469df9e68dde0a60c09b2e9865eda76e9bc24b1cc6b610a2d5406ceef"
EXPECTED_SPLITS = {"train": 3122, "validation": 792, "test": 654}
SPLIT_SEED = "olena-essay-prompt-holdout-v1"   # a mesma divisão por tema do restante da pesquisa

work = Path("essay-work"); work.mkdir(exist_ok=True)
csv_path = work / "essay-br.csv"
if not csv_path.exists():
    url = f"https://raw.githubusercontent.com/rafaelanchieta/essay/{REVISION}/essay-br/essay-br.csv"
    urllib.request.urlretrieve(url, csv_path)
digest = hashlib.sha256(csv_path.read_bytes()).hexdigest()
assert digest == EXPECTED_SHA256, f"Corpus diferente do esperado: {digest}"
print("corpus conferido")

In [ ]:
def fingerprint(text):
    words = re.findall(r"\w+", unicodedata.normalize("NFKC", text).casefold())
    return hashlib.sha256(" ".join(words).encode()).hexdigest()

with csv_path.open(encoding="utf-8", newline="") as stream:
    rows = list(csv.DictReader(stream))

buckets, invalid = defaultdict(list), 0
for index, row in enumerate(rows):
    try:
        paragraphs = ast.literal_eval(row["essay"])
        grades = ast.literal_eval(row["competence"])
        assert isinstance(paragraphs, list) and all(isinstance(p, str) for p in paragraphs)
        assert len(grades) == 5 and all(type(g) is int and g in range(0, 201, 40) for g in grades)
        assert sum(grades) == int(row["score"])
        text = "\n\n".join(p.strip() for p in paragraphs if p.strip())
        prompt = unicodedata.normalize("NFKC", row["prompt"]).strip().casefold()
        assert text and prompt
    except (AssertionError, ValueError, SyntaxError, TypeError, KeyError):
        invalid += 1
        continue
    buckets[fingerprint(text)].append({"text": text, "prompt": prompt, "grades": grades})

kept = []
for candidates in buckets.values():
    if len({(c["prompt"], tuple(c["grades"])) for c in candidates}) != 1:
        continue  # notas ou temas em conflito: excluídos, não misturados
    kept.append(candidates[0])

prompts = sorted({r["prompt"] for r in kept}, key=lambda p: hashlib.sha256(f"{SPLIT_SEED}:{p}".encode()).hexdigest())
train_end, validation_end = round(len(prompts) * .7), round(len(prompts) * .85)
assign = {p: "train" if i < train_end else "validation" if i < validation_end else "test" for i, p in enumerate(prompts)}
for r in kept:
    r["split"] = assign[r["prompt"]]
train = [r for r in kept if r["split"] == "train"]
val = [r for r in kept if r["split"] == "validation"]
test = [r for r in kept if r["split"] == "test"]
counts = {"train": len(train), "validation": len(val), "test": len(test)}
print(counts)
assert counts == EXPECTED_SPLITS, "A divisão não bate com a da pesquisa; não compare com os resultados anteriores."
assert not {r["prompt"] for r in train} & {r["prompt"] for r in test}  # temas separados

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)
if device == "cpu" and not SMOKE:
    print("ATENÇÃO: sem GPU isto levará muitas horas. Ative a GPU nas configurações.")
if SMOKE:
    train, val, test, EPOCHS, MAXLEN = train[:32], val[:16], test[:16], 1, 128

tok = AutoTokenizer.from_pretrained(MODEL)

class Grader(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL)
        self.head = torch.nn.Linear(self.enc.config.hidden_size, 5)
    def forward(self, **batch):
        hidden = self.enc(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        return self.head((hidden * mask).sum(1) / mask.sum(1))

model = Grader().to(device)

def batches(data, size, shuffle):
    order = list(range(len(data)))
    if shuffle: random.shuffle(order)
    for i in range(0, len(order), size):
        chunk = [data[j] for j in order[i:i + size]]
        enc = tok([c["text"] for c in chunk], truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
        target = torch.tensor([[g / 200 for g in c["grades"]] for c in chunk], dtype=torch.float)
        yield {k: v.to(device) for k, v in enc.items()}, target.to(device)

def predict(data):
    model.eval(); out = []
    with torch.no_grad():
        for enc, _ in batches(data, 16, False):
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
                out.append(model(**enc).float().cpu().numpy())
    return np.clip(np.concatenate(out) * 200, 0, 200)

def snap(p): return (np.clip(np.round(p / 40), 0, 5) * 40).astype(int)

def metrics(pred, data):
    y = np.array([r["grades"] for r in data]); p = snap(pred)
    qwk = [float(cohen_kappa_score(y[:, k] // 40, p[:, k] // 40, weights="quadratic")) for k in range(5)]
    return {"qwk": qwk, "qwk_medio": float(np.mean(qwk)),
            "mae_competencia": [float(m) for m in np.abs(y - p).mean(0)],
            "mae_total": float(np.abs(y.sum(1) - p.sum(1)).mean())}

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min(1.0, (s + 1) / (0.1 * steps)) * max(0.0, (steps - s) / steps))
scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

history, best_score, best_state, step, t0 = [], -9.0, None, 0, time.time()
for epoch in range(EPOCHS):
    model.train()
    for enc, target in batches(train, BATCH, True):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
            loss = torch.nn.functional.mse_loss(model(**enc).float(), target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad()
        step += 1
        if step % 25 == 0:
            print(f"época {epoch + 1}/{EPOCHS} passo {step}/{steps} perda {loss.item():.4f} {time.time() - t0:.0f}s", flush=True)
    m = metrics(predict(val), val)
    history.append({"epoca": epoch + 1, **m})
    print(f"validação, época {epoch + 1}: QWK médio {m['qwk_medio']:.3f} | MAE total {m['mae_total']:.1f}", flush=True)
    if m["qwk_medio"] > best_score:   # a época é escolhida SÓ pela validação
        best_score = m["qwk_medio"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
model.load_state_dict(best_state)
print("melhor época:", best_epoch)

In [ ]:
median = np.median([r["grades"] for r in train], axis=0)
baseline = metrics(np.tile(median, (len(test), 1)), test)
final = metrics(predict(test), test)

previous = {  # rodadas anteriores, mesma divisão (docs/research/ESSAY_OWN_GRADER.md)
    "Estrutura (3.122 redações)": ([0.150, 0.362, 0.356, 0.358, 0.383], 115.84),
    "Semântico congelado + estrutura (3.122)": ([0.293, 0.422, 0.279, 0.373, 0.389], 119.51),
    "BERTimbau ajustado, CPU (400 redações)": ([0.254, 0.370, 0.247, 0.323, 0.304], 125.02),
}
line = lambda name, q, mae: f"| {name} | " + " | ".join(f"{x:.3f}" for x in q) + f" | {sum(q) / 5:.3f} | {mae:.1f} |"
print("| Modelo | C1 | C2 | C3 | C4 | C5 | QWK médio | MAE total |")
print("|---|---:|---:|---:|---:|---:|---:|---:|")
print(line("Mediana do treino", baseline["qwk"], baseline["mae_total"]))
for name, (q, mae) in previous.items():
    print(line(name, q, mae))
print(line(f"**Este caderno ({len(train)} redações, {MAXLEN} tokens, época {best_epoch})**", final["qwk"], final["mae_total"]))
print("\nA literatura reporta QWK de 0,60 a 0,73 por competência (outro corpus/divisão, não comparável diretamente).")

Path("resultado.json").write_text(json.dumps({
    "modelo": MODEL, "redacoes_treino": len(train), "max_tokens": MAXLEN, "epocas": EPOCHS,
    "melhor_epoca": best_epoch, "dispositivo": device, "semente": SEED, "smoke": SMOKE,
    "teste": final, "mediana": baseline, "validacao_por_epoca": history,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("gravado: resultado.json (sem texto de redação)")